### Getting end of season awards + the defensive team selections for first and second team from html tables in Basketball Reference

In [37]:
import pandas as pd
import requests
import certifi
import os

from io import StringIO

awards_url = "https://www.basketball-reference.com/awards/awards_2026.html"

response = requests.get(
    awards_url,
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

awards_tables = pd.read_html(
    StringIO(response.content.decode("utf-8"))
)

len(awards_tables)

5

In [38]:
print(f"Number of tables: {len(awards_tables)}")

for i, df in enumerate(awards_tables):
    print(i, df.shape, df.columns.tolist())

Number of tables: 5
0 (8, 20) [('Unnamed: 0_level_0', 'Rank'), ('Unnamed: 1_level_0', 'Player'), ('Unnamed: 2_level_0', 'Age'), ('Unnamed: 3_level_0', 'Tm'), ('Voting', 'First'), ('Voting', 'Pts Won'), ('Voting', 'Pts Max'), ('Voting', 'Share'), ('Unnamed: 8_level_0', 'G'), ('Per Game', 'MP'), ('Per Game', 'PTS'), ('Per Game', 'TRB'), ('Per Game', 'AST'), ('Per Game', 'STL'), ('Per Game', 'BLK'), ('Shooting', 'FG%'), ('Shooting', '3P%'), ('Shooting', 'FT%'), ('Advanced', 'WS'), ('Advanced', 'WS/48')]
1 (5, 20) [('Unnamed: 0_level_0', 'Rank'), ('Unnamed: 1_level_0', 'Player'), ('Unnamed: 2_level_0', 'Age'), ('Unnamed: 3_level_0', 'Tm'), ('Voting', 'First'), ('Voting', 'Pts Won'), ('Voting', 'Pts Max'), ('Voting', 'Share'), ('Unnamed: 8_level_0', 'G'), ('Per Game', 'MP'), ('Per Game', 'PTS'), ('Per Game', 'TRB'), ('Per Game', 'AST'), ('Per Game', 'STL'), ('Per Game', 'BLK'), ('Shooting', 'FG%'), ('Shooting', '3P%'), ('Shooting', 'FT%'), ('Advanced', 'WS'), ('Advanced', 'WS/48')]
2 (31, 2

In [39]:
from bs4 import BeautifulSoup, Comment

html = response.content.decode("utf-8")
soup = BeautifulSoup(html, "html.parser")

visible_tables = soup.find_all("table")
comments = soup.find_all(string=lambda text: isinstance(text, Comment))

commented_tables = sum(
    len(BeautifulSoup(comment, "html.parser").find_all("table"))
    for comment in comments
)

print("Visible HTML tables:", len(visible_tables))
print("Tables inside comments:", commented_tables)
print("Pandas DataFrames:", len(awards_tables))

Visible HTML tables: 5
Tables inside comments: 5
Pandas DataFrames: 5


In [40]:
commented_awards = []

for comment in comments:
    tables = BeautifulSoup(comment, "html.parser").find_all("table")

    for table in tables:
        df = pd.read_html(StringIO(str(table)))[0]
        commented_awards.append(df)

In [41]:
all_awards_tables = awards_tables + commented_awards

print(len(all_awards_tables))

10


In [42]:
for i, df in enumerate(all_awards_tables):
    print(i, df.shape, df.columns.tolist())

0 (8, 20) [('Unnamed: 0_level_0', 'Rank'), ('Unnamed: 1_level_0', 'Player'), ('Unnamed: 2_level_0', 'Age'), ('Unnamed: 3_level_0', 'Tm'), ('Voting', 'First'), ('Voting', 'Pts Won'), ('Voting', 'Pts Max'), ('Voting', 'Share'), ('Unnamed: 8_level_0', 'G'), ('Per Game', 'MP'), ('Per Game', 'PTS'), ('Per Game', 'TRB'), ('Per Game', 'AST'), ('Per Game', 'STL'), ('Per Game', 'BLK'), ('Shooting', 'FG%'), ('Shooting', '3P%'), ('Shooting', 'FT%'), ('Advanced', 'WS'), ('Advanced', 'WS/48')]
1 (5, 20) [('Unnamed: 0_level_0', 'Rank'), ('Unnamed: 1_level_0', 'Player'), ('Unnamed: 2_level_0', 'Age'), ('Unnamed: 3_level_0', 'Tm'), ('Voting', 'First'), ('Voting', 'Pts Won'), ('Voting', 'Pts Max'), ('Voting', 'Share'), ('Unnamed: 8_level_0', 'G'), ('Per Game', 'MP'), ('Per Game', 'PTS'), ('Per Game', 'TRB'), ('Per Game', 'AST'), ('Per Game', 'STL'), ('Per Game', 'BLK'), ('Shooting', 'FG%'), ('Shooting', '3P%'), ('Shooting', 'FT%'), ('Advanced', 'WS'), ('Advanced', 'WS/48')]
2 (31, 23) [('Unnamed: 0_lev

In [43]:
all_awards_tables[2].head(15)

Unnamed: 0_level_0 Unnamed: 1_level_0       Unnamed: 2_level_0  \
                 # Tm                Pos                   Player   
0                 NaN                NaN                      NaN   
1                  1T                  G  Shai Gilgeous-Alexander   
2                  1T                  C             Nikola Jokić   
3                  1T                  C        Victor Wembanyama   
4                  1T                  G              Luka Dončić   
5                  1T                  G          Cade Cunningham   
6                 NaN                NaN                      NaN   
7                  2T                  F             Jaylen Brown   
8                  2T                  F            Kawhi Leonard   
9                  2T                  G         Donovan Mitchell   
10                 2T                  F             Kevin Durant   
11                 2T                  G            Jalen Brunson   
12                NaN                NaN                      NaN   
13                 3T                  G             Tyrese Maxey   
14                 3T                  G             Jamal Murray   

   Unnamed: 3_level_0 Unnamed: 4_level_0  Voting                               \
                  Age                 Tm Pts Won Pts Max  Share 1st Tm 2nd Tm   
0                 NaN                NaN     NaN     NaN    NaN    NaN    NaN   
1                27.0                OKC   500.0   500.0  1.000  100.0    0.0   
2                30.0                DEN   500.0   500.0  1.000  100.0    0.0   
3                22.0                SAS   498.0   500.0  0.996   99.0    1.0   
4                26.0                LAL   482.0   500.0  0.964   91.0    9.0   
5                24.0                DET   414.0   500.0  0.828   60.0   38.0   
6                 NaN                NaN     NaN     NaN    NaN    NaN    NaN   
7                29.0                BOS   384.0   500.0  0.768   44.0   54.0   
8                34.0                LAC   277.0   500.0  0.554    4.0   81.0   
9                29.0                CLE   276.0   500.0  0.552    2.0   85.0   
10               37.0                HOU   241.0   500.0  0.482    0.0   72.0   
11               29.0                NYK   197.0   500.0  0.394    0.0   49.0   
12                NaN                NaN     NaN     NaN    NaN    NaN    NaN   
13               25.0                PHI   168.0   500.0  0.336    0.0   36.0   
14               28.0                DEN   149.0   500.0  0.298    0.0   27.0   

    ... Per Game                       Shooting               Advanced         
    ...      PTS   TRB   AST  STL  BLK      FG%    3P%    FT%       WS  WS/48  
0   ...      NaN   NaN   NaN  NaN  NaN      NaN    NaN    NaN      NaN    NaN  
1   ...     31.1   4.3   6.6  1.4  0.8    0.553  0.386  0.879     15.2  0.323  
2   ...     27.7  12.9  10.7  1.4  0.8    0.569  0.380  0.831     14.9  0.316  
3   ...     25.0  11.5   3.1  1.0  3.1    0.512  0.349  0.827     10.0  0.257  
4   ...     33.5   7.7   8.3  1.6  0.5    0.476  0.366  0.780      9.5  0.199  
5   ...     23.9   5.5   9.9  1.4  0.8    0.461  0.342  0.812      7.9  0.174  
6   ...      NaN   NaN   NaN  NaN  NaN      NaN    NaN    NaN      NaN    NaN  
7   ...     28.7   6.9   5.1  1.0  0.4    0.477  0.347  0.795      6.9  0.135  
8   ...     27.9   6.4   3.6  1.9  0.4    0.505  0.387  0.892      9.2  0.212  
9   ...     27.9   4.5   5.7  1.5  0.3    0.483  0.364  0.865      8.2  0.169  
10  ...     26.0   5.5   4.8  0.8  0.9    0.520  0.413  0.874     10.7  0.180  
11  ...     26.0   3.3   6.8  0.8  0.1    0.467  0.369  0.841      8.8  0.163  
12  ...      NaN   NaN   NaN  NaN  NaN      NaN    NaN    NaN      NaN    NaN  
13  ...     28.3   4.1   6.6  1.9  0.8    0.462  0.367  0.892      8.7  0.156  
14  ...     25.4   4.4   7.1  0.9  0.4    0.483  0.435  0.887      9.5  0.173  

[15 rows x 23 columns]

In [44]:
# Collect the five visible HTML tables
all_html_tables = soup.find_all("table")

# Add the five tables embedded in HTML comments
for comment in comments:
    comment_soup = BeautifulSoup(comment, "html.parser")
    all_html_tables.extend(comment_soup.find_all("table"))

# Inspect each table's HTML ID
for i, table in enumerate(all_html_tables):
    print(i, table.get("id"))

0 mvp
1 roy
2 leading_all_nba
3 leading_all_defense
4 leading_all_rookie
5 dpoy
6 smoy
7 mip
8 clutch_poy
9 coy


In [45]:
import pandas as pd
import requests
import certifi

from bs4 import BeautifulSoup, Comment
from io import StringIO

# 1. Select the season (change this URL each time)
url = "https://www.basketball-reference.com/awards/awards_2026.html"

# 2. Download the webpage
response = requests.get(
    url,
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

# 3. Decode HTML using UTF-8
html = response.content.decode("utf-8")

# 4. Parse the webpage with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# 5. Find visible HTML tables
all_html_tables = soup.find_all("table")

# 6. Find tables hidden inside HTML comments
comments = soup.find_all(
    string=lambda text: isinstance(text, Comment)
)

for comment in comments:
    comment_soup = BeautifulSoup(comment, "html.parser")
    all_html_tables.extend(comment_soup.find_all("table"))

# 7. Print table IDs to identify awards
for i, table in enumerate(all_html_tables):
    print(i, table.get("id"))

# 8. Convert all HTML tables into pandas DataFrames
awards_tables = []

for table in all_html_tables:
    df = pd.read_html(StringIO(str(table)))[0]
    awards_tables.append(df)

print(f"\nTotal DataFrames extracted: {len(awards_tables)}")

0 mvp
1 roy
2 leading_all_nba
3 leading_all_defense
4 leading_all_rookie
5 dpoy
6 smoy
7 mip
8 clutch_poy
9 coy

Total DataFrames extracted: 10


In [46]:
import pandas as pd
import requests
import certifi

from bs4 import BeautifulSoup, Comment
from io import StringIO

# 1. Select the season (change this URL each time)
url = "https://www.basketball-reference.com/awards/awards_2021.html"

# 2. Download the webpage
response = requests.get(
    url,
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

# 3. Decode HTML using UTF-8
html = response.content.decode("utf-8")

# 4. Parse the webpage with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# 5. Find visible HTML tables
all_html_tables = soup.find_all("table")

# 6. Find tables hidden inside HTML comments
comments = soup.find_all(
    string=lambda text: isinstance(text, Comment)
)

for comment in comments:
    comment_soup = BeautifulSoup(comment, "html.parser")
    all_html_tables.extend(comment_soup.find_all("table"))

# 7. Print table IDs to identify awards
for i, table in enumerate(all_html_tables):
    print(i, table.get("id"))

# 8. Convert all HTML tables into pandas DataFrames
awards_tables = []

for table in all_html_tables:
    df = pd.read_html(StringIO(str(table)))[0]
    awards_tables.append(df)

print(f"\nTotal DataFrames extracted: {len(awards_tables)}")

0 mvp
1 roy
2 leading_all_nba
3 leading_all_defense
4 leading_all_rookie
5 dpoy
6 smoy
7 mip
8 coy

Total DataFrames extracted: 9


In [47]:
import pandas as pd
import requests
import certifi

from bs4 import BeautifulSoup, Comment
from io import StringIO

# 1. Select the season (change this URL each time)
url = "https://www.basketball-reference.com/awards/awards_2018.html"

# 2. Download the webpage
response = requests.get(
    url,
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

# 3. Decode HTML using UTF-8
html = response.content.decode("utf-8")

# 4. Parse the webpage with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# 5. Find visible HTML tables
all_html_tables = soup.find_all("table")

# 6. Find tables hidden inside HTML comments
comments = soup.find_all(
    string=lambda text: isinstance(text, Comment)
)

for comment in comments:
    comment_soup = BeautifulSoup(comment, "html.parser")
    all_html_tables.extend(comment_soup.find_all("table"))

# 7. Print table IDs to identify awards
for i, table in enumerate(all_html_tables):
    print(i, table.get("id"))

# 8. Convert all HTML tables into pandas DataFrames
awards_tables = []

for table in all_html_tables:
    df = pd.read_html(StringIO(str(table)))[0]
    awards_tables.append(df)

print(f"\nTotal DataFrames extracted: {len(awards_tables)}")

0 mvp
1 roy
2 leading_all_nba
3 leading_all_defense
4 leading_all_rookie
5 dpoy
6 smoy
7 mip
8 coy

Total DataFrames extracted: 9


In [48]:
import pandas as pd
import requests
import certifi

from bs4 import BeautifulSoup, Comment
from io import StringIO

# 1. Select the season (change this URL each time)
url = "https://www.basketball-reference.com/awards/awards_2012.html"

# 2. Download the webpage
response = requests.get(
    url,
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

# 3. Decode HTML using UTF-8
html = response.content.decode("utf-8")

# 4. Parse the webpage with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# 5. Find visible HTML tables
all_html_tables = soup.find_all("table")

# 6. Find tables hidden inside HTML comments
comments = soup.find_all(
    string=lambda text: isinstance(text, Comment)
)

for comment in comments:
    comment_soup = BeautifulSoup(comment, "html.parser")
    all_html_tables.extend(comment_soup.find_all("table"))

# 7. Print table IDs to identify awards
for i, table in enumerate(all_html_tables):
    print(i, table.get("id"))

# 8. Convert all HTML tables into pandas DataFrames
awards_tables = []

for table in all_html_tables:
    df = pd.read_html(StringIO(str(table)))[0]
    awards_tables.append(df)

print(f"\nTotal DataFrames extracted: {len(awards_tables)}")

0 mvp
1 roy
2 leading_all_nba
3 leading_all_defense
4 leading_all_rookie
5 dpoy
6 smoy
7 mip
8 coy

Total DataFrames extracted: 9


In [49]:
import pandas as pd
import requests
import certifi

from bs4 import BeautifulSoup, Comment
from io import StringIO

# 1. Select the season (change this URL each time)
url = "https://www.basketball-reference.com/awards/awards_2008.html"

# 2. Download the webpage
response = requests.get(
    url,
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

# 3. Decode HTML using UTF-8
html = response.content.decode("utf-8")

# 4. Parse the webpage with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# 5. Find visible HTML tables
all_html_tables = soup.find_all("table")

# 6. Find tables hidden inside HTML comments
comments = soup.find_all(
    string=lambda text: isinstance(text, Comment)
)

for comment in comments:
    comment_soup = BeautifulSoup(comment, "html.parser")
    all_html_tables.extend(comment_soup.find_all("table"))

# 7. Print table IDs to identify awards
for i, table in enumerate(all_html_tables):
    print(i, table.get("id"))

# 8. Convert all HTML tables into pandas DataFrames
awards_tables = []

for table in all_html_tables:
    df = pd.read_html(StringIO(str(table)))[0]
    awards_tables.append(df)

print(f"\nTotal DataFrames extracted: {len(awards_tables)}")

0 mvp
1 roy
2 leading_all_nba
3 leading_all_defense
4 leading_all_rookie
5 dpoy
6 smoy
7 mip
8 coy

Total DataFrames extracted: 9


In [53]:
import pandas as pd
import requests
import certifi

from bs4 import BeautifulSoup, Comment
from io import StringIO

# 1. Select the season (change this URL each time)
url = "https://www.basketball-reference.com/awards/awards_2001.html"

# 2. Download the webpage
response = requests.get(
    url,
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

# 3. Decode HTML using UTF-8
html = response.content.decode("utf-8")

# 4. Parse the webpage with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# 5. Find visible HTML tables
all_html_tables = soup.find_all("table")

# 6. Find tables hidden inside HTML comments
comments = soup.find_all(
    string=lambda text: isinstance(text, Comment)
)

for comment in comments:
    comment_soup = BeautifulSoup(comment, "html.parser")
    all_html_tables.extend(comment_soup.find_all("table"))

# 7. Print table IDs to identify awards
for i, table in enumerate(all_html_tables):
    print(i, table.get("id"))

# 8. Convert all HTML tables into pandas DataFrames
awards_tables = []

for table in all_html_tables:
    df = pd.read_html(StringIO(str(table)))[0]
    awards_tables.append(df)

print(f"\nTotal DataFrames extracted: {len(awards_tables)}")

0 mvp
1 roy
2 leading_all_nba
3 leading_all_defense
4 leading_all_rookie
5 dpoy
6 smoy
7 mip
8 coy

Total DataFrames extracted: 9


In [54]:
# Identify each table by its HTML ID

mvp_html = next(table for table in all_html_tables if table.get("id") == "mvp")

roy_html = next(table for table in all_html_tables if table.get("id") == "roy")

dpoy_html = next(table for table in all_html_tables if table.get("id") == "dpoy")

defense_html = next(table for table in all_html_tables if table.get("id") == "leading_all_defense")

rookie_html = next(table for table in all_html_tables if table.get("id") == "leading_all_rookie")

# Convert each HTML table into a pandas DataFrame

mvp_df = pd.read_html(StringIO(str(mvp_html)))[0]

roy_df = pd.read_html(StringIO(str(roy_html)))[0]

dpoy_df = pd.read_html(StringIO(str(dpoy_html)))[0]

defense_df = pd.read_html(StringIO(str(defense_html)))[0]

rookie_df = pd.read_html(StringIO(str(rookie_html)))[0]

In [57]:
mvp_df.head()

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
                Rank             Player                Age                 Tm   
0                  1      Allen Iverson                 25                PHI   
1                  2         Tim Duncan                 24                SAS   
2                  3   Shaquille O'Neal                 28                LAL   
3                  4       Chris Webber                 27                SAC   
4                  5      Kevin Garnett                 24                MIN   

  Voting                        Unnamed: 8_level_0 Per Game                   \
   First Pts Won Pts Max  Share                  G       MP   PTS   TRB  AST   
0     93    1121    1240  0.904                 71     42.0  31.1   3.8  4.6   
1     18     706    1240  0.569                 82     38.7  22.2  12.2  3.0   
2      7     578    1240  0.466                 74     39.5  28.7  12.7  3.7   
3      5     521    1240  0.420                 70     40.5  27.1  11.1  4.2   
4      1     151    1240  0.122                 81     39.5  22.0  11.4  5.0   

            Shooting               Advanced         
   STL  BLK      FG%    3P%    FT%       WS  WS/48  
0  2.5  0.3    0.420  0.320  0.814     11.8  0.190  
1  0.9  2.3    0.499  0.259  0.618     13.2  0.200  
2  0.6  2.8    0.572  0.000  0.513     14.9  0.245  
3  1.3  1.7    0.481  0.071  0.703     11.0  0.186  
4  1.4  1.8    0.477  0.288  0.764     11.8  0.176

In [59]:
mvp_df.columns.tolist()

[('Unnamed: 0_level_0', 'Rank'),
 ('Unnamed: 1_level_0', 'Player'),
 ('Unnamed: 2_level_0', 'Age'),
 ('Unnamed: 3_level_0', 'Tm'),
 ('Voting', 'First'),
 ('Voting', 'Pts Won'),
 ('Voting', 'Pts Max'),
 ('Voting', 'Share'),
 ('Unnamed: 8_level_0', 'G'),
 ('Per Game', 'MP'),
 ('Per Game', 'PTS'),
 ('Per Game', 'TRB'),
 ('Per Game', 'AST'),
 ('Per Game', 'STL'),
 ('Per Game', 'BLK'),
 ('Shooting', 'FG%'),
 ('Shooting', '3P%'),
 ('Shooting', 'FT%'),
 ('Advanced', 'WS'),
 ('Advanced', 'WS/48')]

In [60]:
mvp_df.columns = mvp_df.columns.get_level_values(-1)

mvp_df.head()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,1,Allen Iverson,25,PHI,93,1121,1240,0.904,71,42.0,31.1,3.8,4.6,2.5,0.3,0.420,0.320,0.814,11.8,0.190
1,2,Tim Duncan,24,SAS,18,706,1240,0.569,82,38.7,22.2,12.2,3.0,0.9,2.3,0.499,0.259,0.618,13.2,0.200
2,3,Shaquille O'Neal,28,LAL,7,578,1240,0.466,74,39.5,28.7,12.7,3.7,0.6,2.8,0.572,0.000,0.513,14.9,0.245
3,4,Chris Webber,27,SAC,5,521,1240,0.420,70,40.5,27.1,11.1,4.2,1.3,1.7,0.481,0.071,0.703,11.0,0.186
4,5,Kevin Garnett,24,MIN,1,151,1240,0.122,81,39.5,22.0,11.4,5.0,1.4,1.8,0.477,0.288,0.764,11.8,0.176


In [61]:
mvp_df_final = mvp_df[["Rank", "Player", "Age", "Tm", "First", "Pts Won", "Pts Max", "Share"]].copy()

mvp_df_final.head()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share
0,1,Allen Iverson,25,PHI,93,1121,1240,0.904
1,2,Tim Duncan,24,SAS,18,706,1240,0.569
2,3,Shaquille O'Neal,28,LAL,7,578,1240,0.466
3,4,Chris Webber,27,SAC,5,521,1240,0.420
4,5,Kevin Garnett,24,MIN,1,151,1240,0.122


In [ ]:
mvp_final = mvp_df_final.rename(columns={
    "Rank": "voting_rank",
    "Player": "player_name",
    "Age": "age",
    "Tm": "team",
    "First": "first_place_votes",
    "Pts Won": "points_won",
    "Pts Max": "points_max",
    "Share": "vote_share"
}).copy()

mvp_final["season"] = "2000-2001"
mvp_final["season_end_year"] = 2001

mvp_final["award_winner"] = mvp_final["voting_rank"] == 1

In [72]:
mvp_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   voting_rank        17 non-null     str    
 1   player_name        17 non-null     str    
 2   age                17 non-null     int64  
 3   team               17 non-null     str    
 4   first_place_votes  17 non-null     int64  
 5   points_won         17 non-null     int64  
 6   points_max         17 non-null     int64  
 7   vote_share         17 non-null     float64
 8   season             17 non-null     str    
 9   season_end_year    17 non-null     int64  
 10  award_winner       17 non-null     bool   
dtypes: bool(1), float64(1), int64(5), str(4)
memory usage: 1.5 KB


In [73]:
mvp_final["voting_rank"] = (
    mvp_final["voting_rank"]
    .str.replace("T", "", regex=False)
    .astype(int)
)

In [74]:
mvp_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   voting_rank        17 non-null     int64  
 1   player_name        17 non-null     str    
 2   age                17 non-null     int64  
 3   team               17 non-null     str    
 4   first_place_votes  17 non-null     int64  
 5   points_won         17 non-null     int64  
 6   points_max         17 non-null     int64  
 7   vote_share         17 non-null     float64
 8   season             17 non-null     str    
 9   season_end_year    17 non-null     int64  
 10  award_winner       17 non-null     bool   
dtypes: bool(1), float64(1), int64(6), str(3)
memory usage: 1.5 KB


In [75]:
mvp_final.head()

,voting_rank,player_name,age,team,first_place_votes,points_won,points_max,vote_share,season,season_end_year,award_winner
0,1,Allen Iverson,25,PHI,93,1121,1240,0.904,2000-01,2001,False
1,2,Tim Duncan,24,SAS,18,706,1240,0.569,2000-01,2001,False
2,3,Shaquille O'Neal,28,LAL,7,578,1240,0.466,2000-01,2001,False
3,4,Chris Webber,27,SAC,5,521,1240,0.420,2000-01,2001,False
4,5,Kevin Garnett,24,MIN,1,151,1240,0.122,2000-01,2001,False


In [76]:

mvp_final["award_winner"] = mvp_final["voting_rank"] == 1

In [77]:
mvp_final.head()

,voting_rank,player_name,age,team,first_place_votes,points_won,points_max,vote_share,season,season_end_year,award_winner
0,1,Allen Iverson,25,PHI,93,1121,1240,0.904,2000-01,2001,True
1,2,Tim Duncan,24,SAS,18,706,1240,0.569,2000-01,2001,False
2,3,Shaquille O'Neal,28,LAL,7,578,1240,0.466,2000-01,2001,False
3,4,Chris Webber,27,SAC,5,521,1240,0.420,2000-01,2001,False
4,5,Kevin Garnett,24,MIN,1,151,1240,0.122,2000-01,2001,False


In [78]:
roy_df.head()

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
                Rank             Player                Age                 Tm   
0                  1        Mike Miller                 20                ORL   
1                  2      Kenyon Martin                 23                NJN   
2                  3       Marc Jackson                 26                GSW   
3                 4T       Darius Miles                 19                LAC   
4                 4T    Morris Peterson                 23                TOR   

  Voting                        Unnamed: 8_level_0 Per Game                  \
   First Pts Won Pts Max  Share                  G       MP   PTS  TRB  AST   
0     75      75     124  0.605                 82     29.1  11.9  4.0  1.7   
1     36      36     124  0.290                 68     33.4  12.0  7.4  1.9   
2      7       7     124  0.056                 48     29.4  13.2  7.5  1.2   
3      3       3     124  0.024                 81     26.3   9.4  5.9  1.2   
4      3       3     124  0.024                 80     22.6   9.3  3.2  1.3   

            Shooting               Advanced         
   STL  BLK      FG%    3P%    FT%       WS  WS/48  
0  0.6  0.2    0.436  0.407  0.711      5.3  0.106  
1  1.1  1.7    0.445  0.091  0.630      2.1  0.044  
2  0.7  0.6    0.467  0.217  0.802      2.6  0.089  
3  0.6  1.5    0.505  0.053  0.521      3.0  0.068  
4  0.8  0.3    0.431  0.382  0.717      3.6  0.096

In [81]:
roy_df.columns = roy_df.columns.get_level_values(-1)

In [82]:
roy_df.head()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,1,Mike Miller,20,ORL,75,75,124,0.605,82,29.1,11.9,4.0,1.7,0.6,0.2,0.436,0.407,0.711,5.3,0.106
1,2,Kenyon Martin,23,NJN,36,36,124,0.290,68,33.4,12.0,7.4,1.9,1.1,1.7,0.445,0.091,0.630,2.1,0.044
2,3,Marc Jackson,26,GSW,7,7,124,0.056,48,29.4,13.2,7.5,1.2,0.7,0.6,0.467,0.217,0.802,2.6,0.089
3,4T,Darius Miles,19,LAC,3,3,124,0.024,81,26.3,9.4,5.9,1.2,0.6,1.5,0.505,0.053,0.521,3.0,0.068
4,4T,Morris Peterson,23,TOR,3,3,124,0.024,80,22.6,9.3,3.2,1.3,0.8,0.3,0.431,0.382,0.717,3.6,0.096


In [85]:
roy_df_final_cols = roy_df[["Rank", "Player", "Age", "Tm", "First", "Pts Won", "Pts Max", "Share"]].copy()

roy_df_final_cols.head()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share
0,1,Mike Miller,20,ORL,75,75,124,0.605
1,2,Kenyon Martin,23,NJN,36,36,124,0.290
2,3,Marc Jackson,26,GSW,7,7,124,0.056
3,4T,Darius Miles,19,LAC,3,3,124,0.024
4,4T,Morris Peterson,23,TOR,3,3,124,0.024


In [86]:
roy_final = roy_df_final_cols.rename(columns={
    "Rank": "voting_rank",
    "Player": "player_name",
    "Age": "age",
    "Tm": "team",
    "First": "first_place_votes",
    "Pts Won": "points_won",
    "Pts Max": "points_max",
    "Share": "vote_share"
}).copy()

roy_final["voting_rank"] = roy_final["voting_rank"].astype(str).str.replace("T", "", regex=False).astype(int)

roy_final["season"] = "2000-2001"
roy_final["year"] = 2001

roy_final["award_winner"] = roy_final["voting_rank"] == 1

roy_final.head()



,voting_rank,player_name,age,team,first_place_votes,points_won,points_max,vote_share,season,year,award_winner
0,1,Mike Miller,20,ORL,75,75,124,0.605,2000-2001,2001,True
1,2,Kenyon Martin,23,NJN,36,36,124,0.290,2000-2001,2001,False
2,3,Marc Jackson,26,GSW,7,7,124,0.056,2000-2001,2001,False
3,4,Darius Miles,19,LAC,3,3,124,0.024,2000-2001,2001,False
4,4,Morris Peterson,23,TOR,3,3,124,0.024,2000-2001,2001,False


In [87]:
roy_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   voting_rank        5 non-null      int64  
 1   player_name        5 non-null      str    
 2   age                5 non-null      int64  
 3   team               5 non-null      str    
 4   first_place_votes  5 non-null      int64  
 5   points_won         5 non-null      int64  
 6   points_max         5 non-null      int64  
 7   vote_share         5 non-null      float64
 8   season             5 non-null      str    
 9   year               5 non-null      int64  
 10  award_winner       5 non-null      bool   
dtypes: bool(1), float64(1), int64(6), str(3)
memory usage: 537.0 bytes


In [89]:
dpoy_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 23 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   (Unnamed: 0_level_0, Rank)    14 non-null     str    
 1   (Unnamed: 1_level_0, Player)  14 non-null     str    
 2   (Unnamed: 2_level_0, Age)     14 non-null     int64  
 3   (Unnamed: 3_level_0, Tm)      14 non-null     str    
 4   (Voting, First)               14 non-null     int64  
 5   (Voting, Pts Won)             14 non-null     int64  
 6   (Voting, Pts Max)             14 non-null     int64  
 7   (Voting, Share)               14 non-null     float64
 8   (Unnamed: 8_level_0, G)       14 non-null     int64  
 9   (Per Game, MP)                14 non-null     float64
 10  (Per Game, PTS)               14 non-null     float64
 11  (Per Game, TRB)               14 non-null     float64
 12  (Per Game, AST)               14 non-null     float64
 13  (Per Game, STL)   

In [90]:
dpoy_df.columns = dpoy_df.columns.get_level_values(-1)

dpoy_df_final_cols = dpoy_df[["Rank", "Player", "Age", "Tm", "First", "Pts Won", "Pts Max", "Share"]]

dpoy_df_final_cols.head()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share
0,1,Dikembe Mutombo,34,TOT,48,48,123,0.390
1,2,Kevin Garnett,24,MIN,26,26,123,0.211
2,3,Tim Duncan,24,SAS,14,14,123,0.114
3,4,Doug Christie,30,SAC,9,9,123,0.073
4,5T,David Robinson,35,SAS,6,6,123,0.049


In [91]:
dpoy_final = dpoy_df_final_cols.rename(columns={
    "Rank": "voting_rank",
    "Player": "player_name",
    "Age": "age",
    "Tm": "team",
    "First": "first_place_votes",
    "Pts Won": "points_won",
    "Pts Max": "points_max",
    "Share": "vote_share"
}).copy()

dpoy_final["voting_rank"] = dpoy_final["voting_rank"].astype(str).str.replace("T","",regex=False).astype(int)

dpoy_final["season"] = "2000-2001"
dpoy_final["year"] = 2001

dpoy_final["award_winner"] = dpoy_final["voting_rank"] == 1

dpoy_final.head()

,voting_rank,player_name,age,team,first_place_votes,points_won,points_max,vote_share,season,year,award_winner
0,1,Dikembe Mutombo,34,TOT,48,48,123,0.390,2000-2001,2001,True
1,2,Kevin Garnett,24,MIN,26,26,123,0.211,2000-2001,2001,False
2,3,Tim Duncan,24,SAS,14,14,123,0.114,2000-2001,2001,False
3,4,Doug Christie,30,SAC,9,9,123,0.073,2000-2001,2001,False
4,5,David Robinson,35,SAS,6,6,123,0.049,2000-2001,2001,False


In [92]:
dpoy_final.isna().sum()

voting_rank          0
player_name          0
age                  0
team                 0
first_place_votes    0
points_won           0
points_max           0
vote_share           0
season               0
year                 0
award_winner         0
dtype: int64

In [97]:
rookie_df.head(12)

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                 # Tm             Player                Age   
0                 1st        Mike Miller               20.0   
1                 1st      Kenyon Martin               23.0   
2                 1st       Marc Jackson               26.0   
3                 1st    Morris Peterson               23.0   
4                 1st       Darius Miles               19.0   
5                 NaN                NaN                NaN   
6                 2nd      Hedo Türkoğlu               21.0   
7                 2nd      Desmond Mason               23.0   
8                 2nd       Marcus Fizer               22.0   
9                 2nd         Chris Mihm               21.0   
10                NaN                NaN                NaN   
11                ORV    Jamaal Magloire               22.0   

   Unnamed: 3_level_0  Voting                              Unnamed: 9_level_0  \
                   Tm Pts Won Pts Max  Share 1st Tm 2nd Tm                  G   
0                 ORL    56.0    56.0  1.000   28.0    0.0               82.0   
1                 NJN    54.0    56.0  0.964   27.0    0.0               68.0   
2                 GSW    51.0    56.0  0.911   24.0    3.0               48.0   
3                 TOR    50.0    56.0  0.893   22.0    6.0               80.0   
4                 LAC    48.0    56.0  0.857   21.0    6.0               81.0   
5                 NaN     NaN     NaN    NaN    NaN    NaN                NaN   
6                 SAC    34.0    56.0  0.607    9.0   16.0               74.0   
7                 SEA    28.0    56.0  0.500    5.0   18.0               78.0   
8                 CHI    16.0    56.0  0.286    1.0   14.0               72.0   
9                 CLE    13.0    56.0  0.232    1.0   11.0               59.0   
10                NaN     NaN     NaN    NaN    NaN    NaN                NaN   
11                CHH    12.0    56.0  0.214    0.0   12.0               74.0   

    ... Per Game                     Shooting               Advanced         
    ...      PTS  TRB  AST  STL  BLK      FG%    3P%    FT%       WS  WS/48  
0   ...     11.9  4.0  1.7  0.6  0.2    0.436  0.407  0.711      5.3  0.106  
1   ...     12.0  7.4  1.9  1.1  1.7    0.445  0.091  0.630      2.1  0.044  
2   ...     13.2  7.5  1.2  0.7  0.6    0.467  0.217  0.802      2.6  0.089  
3   ...      9.3  3.2  1.3  0.8  0.3    0.431  0.382  0.717      3.6  0.096  
4   ...      9.4  5.9  1.2  0.6  1.5    0.505  0.053  0.521      3.0  0.068  
5   ...      NaN  NaN  NaN  NaN  NaN      NaN    NaN    NaN      NaN    NaN  
6   ...      5.3  2.8  0.9  0.7  0.3    0.412  0.326  0.777      2.8  0.106  
7   ...      5.9  3.2  0.8  0.5  0.3    0.431  0.269  0.736      1.7  0.055  
8   ...      9.5  4.3  1.1  0.4  0.3    0.430  0.256  0.727     -0.7 -0.022  
9   ...      7.6  4.7  0.3  0.3  0.9    0.442  0.000  0.794      1.3  0.053  
10  ...      NaN  NaN  NaN  NaN  NaN      NaN    NaN    NaN      NaN    NaN  
11  ...      4.6  4.0  0.4  0.2  1.1    0.450  0.000  0.655      2.6  0.114  

[12 rows x 21 columns]

In [96]:
defense_df.head(12)

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                 # Tm                Pos             Player   
0                 1st                  C    Dikembe Mutombo   
1                 1st                  F         Tim Duncan   
2                 1st                  G         Jason Kidd   
3                 1st                  G        Gary Payton   
4                 1st                  F      Kevin Garnett   
5                 NaN                NaN                NaN   
6                 2nd                  G        Kobe Bryant   
7                 2nd                  G      Doug Christie   
8                 2nd                  F        Bruce Bowen   
9                 2nd                  C   Shaquille O'Neal   
10                2nd                  F         P.J. Brown   
11                NaN                NaN                NaN   

   Unnamed: 3_level_0 Unnamed: 4_level_0  Voting                 \
                  Age                 Tm Pts Won Pts Max  Share   
0                34.0                TOT    43.0    58.0  0.741   
1                24.0                SAS    40.0    58.0  0.690   
2                27.0                PHO    39.0    58.0  0.672   
3                32.0                SEA    38.0    58.0  0.655   
4                24.0                MIN    24.0    58.0  0.414   
5                 NaN                NaN     NaN     NaN    NaN   
6                22.0                LAL    26.0    58.0  0.448   
7                30.0                SAC    20.0    58.0  0.345   
8                29.0                MIA    18.0    58.0  0.310   
9                28.0                LAL    17.0    58.0  0.293   
10               31.0                CHH    13.0    58.0  0.224   
11                NaN                NaN     NaN     NaN    NaN   

   Unnamed: 8_level_0 Per Game  ...           Shooting               Advanced  \
                    G       MP  ...  STL  BLK      FG%    3P%    FT%       WS   
0                75.0     34.5  ...  0.4  2.7    0.484    NaN  0.725      8.1   
1                82.0     38.7  ...  0.9  2.3    0.499  0.259  0.618     13.2   
2                77.0     39.8  ...  2.2  0.3    0.411  0.297  0.814      9.6   
3                79.0     41.1  ...  1.6  0.3    0.456  0.375  0.766     10.8   
4                81.0     39.5  ...  1.4  1.8    0.477  0.288  0.764     11.8   
5                 NaN      NaN  ...  NaN  NaN      NaN    NaN    NaN      NaN   
6                68.0     40.9  ...  1.7  0.6    0.464  0.305  0.853     11.3   
7                81.0     36.3  ...  2.3  0.6    0.395  0.376  0.897      8.7   
8                82.0     32.7  ...  1.0  0.6    0.363  0.336  0.609      4.2   
9                74.0     39.5  ...  0.6  2.8    0.572  0.000  0.513     14.9   
10               80.0     35.1  ...  1.0  1.2    0.444  0.000  0.852      8.0   
11                NaN      NaN  ...  NaN  NaN      NaN    NaN    NaN      NaN   

                            
    WS/48  DWS DBPM   DRtg  
0   0.151  4.7  1.3   97.0  
1   0.200  7.1  1.8   94.0  
2   0.150  5.4  1.7   97.0  
3   0.160  1.7 -0.7  107.0  
4   0.176  4.8  1.4   99.0  
5     NaN  NaN  NaN    NaN  
6   0.196  2.2 -0.3  105.0  
7   0.141  4.4  2.3   99.0  
8   0.075  3.7  1.5  100.0  
9   0.245  3.9  0.7  101.0  
10  0.137  4.7  0.9   98.0  
11    NaN  NaN  NaN    NaN  

[12 rows x 23 columns]

In [101]:
rookie_df.columns = rookie_df.columns.get_level_values(-1)

defense_df.columns = defense_df.columns.get_level_values(-1)

rookie_df.head(15)

,# Tm,Player,Age,Tm,Pts Won,Pts Max,Share,1st Tm,2nd Tm,G,...,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,1st,Mike Miller,20.0,ORL,56.0,56.0,1.000,28.0,0.0,82.0,...,11.9,4.0,1.7,0.6,0.2,0.436,0.407,0.711,5.3,0.106
1,1st,Kenyon Martin,23.0,NJN,54.0,56.0,0.964,27.0,0.0,68.0,...,12.0,7.4,1.9,1.1,1.7,0.445,0.091,0.630,2.1,0.044
2,1st,Marc Jackson,26.0,GSW,51.0,56.0,0.911,24.0,3.0,48.0,...,13.2,7.5,1.2,0.7,0.6,0.467,0.217,0.802,2.6,0.089
3,1st,Morris Peterson,23.0,TOR,50.0,56.0,0.893,22.0,6.0,80.0,...,9.3,3.2,1.3,0.8,0.3,0.431,0.382,0.717,3.6,0.096
4,1st,Darius Miles,19.0,LAC,48.0,56.0,0.857,21.0,6.0,81.0,...,9.4,5.9,1.2,0.6,1.5,0.505,0.053,0.521,3.0,0.068
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2nd,Hedo Türkoğlu,21.0,SAC,34.0,56.0,0.607,9.0,16.0,74.0,...,5.3,2.8,0.9,0.7,0.3,0.412,0.326,0.777,2.8,0.106
7,2nd,Desmond Mason,23.0,SEA,28.0,56.0,0.500,5.0,18.0,78.0,...,5.9,3.2,0.8,0.5,0.3,0.431,0.269,0.736,1.7,0.055
8,2nd,Marcus Fizer,22.0,CHI,16.0,56.0,0.286,1.0,14.0,72.0,...,9.5,4.3,1.1,0.4,0.3,0.430,0.256,0.727,-0.7,-0.022
9,2nd,Chris Mihm,21.0,CLE,13.0,56.0,0.232,1.0,11.0,59.0,...,7.6,4.7,0.3,0.3,0.9,0.442,0.000,0.794,1.3,0.053


In [104]:
rookie_df_cleaned = rookie_df[["# Tm", "Player", "Age", "Tm", "Pts Won", "Pts Max", "Share", "1st Tm", "2nd Tm"]].copy()

defense_df_cleaned =defense_df[["# Tm", "Player", "Age", "Tm", "Pts Won", "Pts Max", "Share"]].copy()

defense_df_cleaned.head()


,# Tm,Player,Age,Tm,Pts Won,Pts Max,Share
0,1st,Dikembe Mutombo,34.0,TOT,43.0,58.0,0.741
1,1st,Tim Duncan,24.0,SAS,40.0,58.0,0.690
2,1st,Jason Kidd,27.0,PHO,39.0,58.0,0.672
3,1st,Gary Payton,32.0,SEA,38.0,58.0,0.655
4,1st,Kevin Garnett,24.0,MIN,24.0,58.0,0.414


In [110]:
rookie_df_cleaner = rookie_df_cleaned[rookie_df_cleaned["# Tm"].notna()].copy()
defense_df_cleaner = defense_df_cleaned[defense_df_cleaned["# Tm"].notna()].copy()

defense_df_cleaner.head(15)

,# Tm,Player,Age,Tm,Pts Won,Pts Max,Share
0,1st,Dikembe Mutombo,34.0,TOT,43.0,58.0,0.741
1,1st,Tim Duncan,24.0,SAS,40.0,58.0,0.690
2,1st,Jason Kidd,27.0,PHO,39.0,58.0,0.672
3,1st,Gary Payton,32.0,SEA,38.0,58.0,0.655
4,1st,Kevin Garnett,24.0,MIN,24.0,58.0,0.414
6,2nd,Kobe Bryant,22.0,LAL,26.0,58.0,0.448
7,2nd,Doug Christie,30.0,SAC,20.0,58.0,0.345
8,2nd,Bruce Bowen,29.0,MIA,18.0,58.0,0.310
9,2nd,Shaquille O'Neal,28.0,LAL,17.0,58.0,0.293
10,2nd,P.J. Brown,31.0,CHH,13.0,58.0,0.224


In [114]:
rookie_df_cleaner.info()

<class 'pandas.DataFrame'>
Index: 21 entries, 0 to 22
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   # Tm     21 non-null     str    
 1   Player   21 non-null     str    
 2   Age      21 non-null     float64
 3   Tm       21 non-null     str    
 4   Pts Won  21 non-null     float64
 5   Pts Max  21 non-null     float64
 6   Share    21 non-null     float64
 7   1st Tm   21 non-null     float64
 8   2nd Tm   21 non-null     float64
dtypes: float64(6), str(3)
memory usage: 1.6 KB


In [116]:
rookie_final = rookie_df_cleaner.rename(columns={
    "# Tm": "team_selection",
    "Player": "player_name",
    "Age": "age",
    "Tm": "team",
    "Pts Won": "points_won",
    "Pts Max": "points_max",
    "Share": "vote_share",
    "1st Tm": "first_team_votes",
    "2nd Tm": "second_team_votes"
}).copy()

defense_final = defense_df_cleaner.rename(columns={
    "# Tm": "team_selection",
    "Pos": "position",
    "Player": "player_name",
    "Age": "age",
    "Tm": "team",
    "Pts Won": "points_won",
    "Pts Max": "points_max",
    "Share": "vote_share"
}).copy()

rookie_final["season"] = "2000-2001"
rookie_final["year"] = 2001

defense_final["season"] = "2000-2001"
defense_final["year"] = 2001

rookie_final[[
    "age",
    "points_won",
    "points_max",
    "first_team_votes",
    "second_team_votes"
]] = rookie_final[[
    "age",
    "points_won",
    "points_max",
    "first_team_votes",
    "second_team_votes"
]].astype(int)

defense_final[[
    "age",
    "points_won",
    "points_max"
]] = defense_final[[
    "age",
    "points_won",
    "points_max"
]].astype(int)



In [120]:
rookie_final.info()

<class 'pandas.DataFrame'>
Index: 21 entries, 0 to 22
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   team_selection     21 non-null     str    
 1   player_name        21 non-null     str    
 2   age                21 non-null     int64  
 3   team               21 non-null     str    
 4   points_won         21 non-null     int64  
 5   points_max         21 non-null     int64  
 6   vote_share         21 non-null     float64
 7   first_team_votes   21 non-null     int64  
 8   second_team_votes  21 non-null     int64  
 9   season             21 non-null     str    
 10  year               21 non-null     int64  
dtypes: float64(1), int64(6), str(4)
memory usage: 2.0 KB


In [121]:

import pandas as pd
import requests
import certifi
from bs4 import BeautifulSoup, Comment
from io import StringIO

def extract_awards(year):

    # STAGE 1: Construct the URL
    url = f"https://www.basketball-reference.com/awards/awards_{year}.html"

    # STAGE 2: Download and parse the webpage
    response = requests.get(url, verify=certifi.where(), timeout=20)
    response.raise_for_status()

    html = response.content.decode("utf-8")
    soup = BeautifulSoup(html, "html.parser")

    # STAGE 3: Collect regular and commented-out HTML tables
    all_tables = soup.find_all("table")

    comments = soup.find_all(
        string=lambda text: isinstance(text, Comment)
    )

    for comment in comments:
        comment_soup = BeautifulSoup(comment, "html.parser")
        all_tables.extend(comment_soup.find_all("table"))

    # STAGE 4: Identify the five award tables
    table_ids = {
        "mvp": "mvp",
        "roy": "roy",
        "dpoy": "dpoy",
        "rookie": "leading_all_rookie",
        "defense": "leading_all_defense"
    }

    # STAGE 5: Convert each HTML table into a DataFrame
    extracted = {}

    for name, table_id in table_ids.items():

        table = next(
            table for table in all_tables
            if table.get("id") == table_id
        )

        extracted[name] = pd.read_html(StringIO(str(table)))[0]

    # STAGE 6: Return the five DataFrames
    return extracted

awards_2013 = extract_awards(2013)

In [122]:
awards_2013

{'mvp':    Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                  Rank             Player                Age   
 0                   1       LeBron James                 28   
 1                   2       Kevin Durant                 24   
 2                   3    Carmelo Anthony                 28   
 3                   4         Chris Paul                 27   
 4                   5        Kobe Bryant                 34   
 5                   6        Tony Parker                 30   
 6                   7         Tim Duncan                 36   
 7                   8       James Harden                 23   
 8                   9  Russell Westbrook                 24   
 9                  10        Dwyane Wade                 31   
 10                 11      Stephen Curry                 24   
 11                12T      Kevin Garnett                 36   
 12                12T         Marc Gasol                 28   
 13                12T          T

In [124]:
awards_2026 = extract_awards(2026)

awards_2026

{'mvp':   Unnamed: 0_level_0       Unnamed: 1_level_0 Unnamed: 2_level_0  \
                 Rank                   Player                Age   
 0                  1  Shai Gilgeous-Alexander                 27   
 1                  2             Nikola Jokić                 30   
 2                  3        Victor Wembanyama                 22   
 3                  4              Luka Dončić                 26   
 4                  5          Cade Cunningham                 24   
 5                  6             Jaylen Brown                 29   
 6                 7T            Kawhi Leonard                 34   
 7                 7T         Donovan Mitchell                 29   
 
   Unnamed: 3_level_0 Voting                        Unnamed: 8_level_0  \
                   Tm  First Pts Won Pts Max  Share                  G   
 0                OKC     83     939    1000  0.939                 68   
 1                DEN     10     634    1000  0.634                 65   
 2   

In [132]:
awards_2013["mvp"].columns.to_list()

[('Unnamed: 0_level_0', 'Rank'),
 ('Unnamed: 1_level_0', 'Player'),
 ('Unnamed: 2_level_0', 'Age'),
 ('Unnamed: 3_level_0', 'Tm'),
 ('Voting', 'First'),
 ('Voting', 'Pts Won'),
 ('Voting', 'Pts Max'),
 ('Voting', 'Share'),
 ('Unnamed: 8_level_0', 'G'),
 ('Per Game', 'MP'),
 ('Per Game', 'PTS'),
 ('Per Game', 'TRB'),
 ('Per Game', 'AST'),
 ('Per Game', 'STL'),
 ('Per Game', 'BLK'),
 ('Shooting', 'FG%'),
 ('Shooting', '3P%'),
 ('Shooting', 'FT%'),
 ('Advanced', 'WS'),
 ('Advanced', 'WS/48')]

In [149]:
### testing for only the 2001 season...



import pandas as pd
import requests
import certifi
from bs4 import BeautifulSoup, Comment
from io import StringIO


# FUNCTION 1: EXTRACT THE FIVE RAW AWARDS TABLES

def extract_awards(year):

    # A. Build URL and retrieve HTML
    url = f"https://www.basketball-reference.com/awards/awards_{year}.html"

    response = requests.get(url, verify=certifi.where(), timeout=20)
    response.raise_for_status()

    html = response.content.decode("utf-8")
    soup = BeautifulSoup(html, "html.parser")

    # B. Find visible and commented-out HTML tables
    all_tables = soup.find_all("table")

    comments = soup.find_all(
        string=lambda text: isinstance(text, Comment)
    )

    for comment in comments:
        comment_soup = BeautifulSoup(comment, "html.parser")
        all_tables.extend(comment_soup.find_all("table"))

    # C. Map award names to HTML table IDs
    table_ids = {
        "mvp": "mvp",
        "roy": "roy",
        "dpoy": "dpoy",
        "rookie": "leading_all_rookie",
        "defense": "leading_all_defense"
    }

    # D. Extract the five tables into DataFrames
    extracted = {}

    for name, table_id in table_ids.items():

        table = next(
            table for table in all_tables
            if table.get("id") == table_id
        )

        extracted[name] = pd.read_html(StringIO(str(table)))[0]

    return extracted


# FUNCTION 2: CLEAN THE FIVE DATAFRAMES

def clean_awards(raw_tables, year):

    cleaned = {}

    # A. Clean individual awards: MVP, ROY, DPOY
    for award in ["mvp", "roy", "dpoy"]:

        df = raw_tables[award].copy()
        df.columns = df.columns.get_level_values(-1)

        df = df[[
            "Rank", "Player", "Age", "Tm",
            "First", "Pts Won", "Pts Max", "Share"
        ]].copy()

        df = df.rename(columns={
            "Rank": "voting_rank",
            "Player": "player_name",
            "Age": "age",
            "Tm": "team",
            "First": "first_place_votes",
            "Pts Won": "points_won",
            "Pts Max": "points_max",
            "Share": "vote_share"
        })

        df["voting_rank"] = (
            df["voting_rank"]
            .astype(str)
            .str.replace("T", "", regex=False)
            .astype(int)
        )

        integer_columns = [
            "age", "first_place_votes",
            "points_won", "points_max"
        ]

        df[integer_columns] = df[integer_columns].astype(int)

        df["season"] = f"{year-1}-{year}"
        df["year"] = year
        df["award_winner"] = df["voting_rank"] == 1

        cleaned[award] = df

    # B. Clean team awards: All-Rookie, All-Defensive
    for award in ["rookie", "defense"]:

        df = raw_tables[award].copy()
        df.columns = df.columns.get_level_values(-1)

        df = df[df["# Tm"].isin(["1st", "2nd"])].copy()

        columns = [
            "# Tm", "Player", "Age", "Tm",
            "Pts Won", "Pts Max", "Share"
        ]

        if award == "rookie":
            columns += ["1st Tm", "2nd Tm"]
        else:
            columns += ["Pos"]

        df = df[columns].copy()

        df = df.rename(columns={
            "# Tm": "team_selection",
            "Player": "player_name",
            "Age": "age",
            "Tm": "team",
            "Pts Won": "points_won",
            "Pts Max": "points_max",
            "Share": "vote_share",
            "1st Tm": "first_team_votes",
            "2nd Tm": "second_team_votes",
            "Pos": "position"
        })

        integer_columns = ["age", "points_won", "points_max"]

        if award == "rookie":
            integer_columns += ["first_team_votes", "second_team_votes"]

        df[integer_columns] = df[integer_columns].astype(int)

        df["season"] = f"{year-1}-{year}"
        df["year"] = year

        cleaned[award] = df

    return cleaned


# RUN BOTH FUNCTIONS FOR 2001

raw_2001 = extract_awards(2001)
clean_2001 = clean_awards(raw_2001, 2001)


In [138]:
clean_2001["roy"].head()

,voting_rank,player_name,age,team,first_place_votes,points_won,points_max,vote_share,season,year,award_winner
0,1,Mike Miller,20,ORL,75,75,124,0.605,2000-2001,2001,True
1,2,Kenyon Martin,23,NJN,36,36,124,0.290,2000-2001,2001,False
2,3,Marc Jackson,26,GSW,7,7,124,0.056,2000-2001,2001,False
3,4,Darius Miles,19,LAC,3,3,124,0.024,2000-2001,2001,False
4,4,Morris Peterson,23,TOR,3,3,124,0.024,2000-2001,2001,False


In [146]:
raw_2026 = extract_awards(2026)
clean_2026 = clean_awards(raw_2026, 2026)

In [148]:
clean_2026["mvp"].head()

,voting_rank,player_name,age,team,first_place_votes,points_won,points_max,vote_share,season,year,award_winner
0,1,Shai Gilgeous-Alexander,27,OKC,83,939,1000,0.939,2025-2026,2026,True
1,2,Nikola Jokić,30,DEN,10,634,1000,0.634,2025-2026,2026,False
2,3,Victor Wembanyama,22,SAS,5,569,1000,0.569,2025-2026,2026,False
3,4,Luka Dončić,26,LAL,0,250,1000,0.250,2025-2026,2026,False
4,5,Cade Cunningham,24,DET,2,117,1000,0.117,2025-2026,2026,False


In [166]:

from pathlib import Path
import time

def build_awards_history(start_year=2001, end_year=2026):

    award_names = ["mvp", "roy", "dpoy", "rookie", "defense"]

    # Track cleaned DataFrames for final concatenation
    cleaned_history = {award: [] for award in award_names}

    failed_years = []

    for year in range(start_year, end_year + 1):

        print(f"Processing {year}...")

        raw_folder = PROJECT_ROOT / "data" / "raw" / "awards" / str(year)
        staging_folder = PROJECT_ROOT / "data" / "staging" / "awards" / str(year)


        raw_folder.mkdir(parents=True, exist_ok=True)
        staging_folder.mkdir(parents=True, exist_ok=True)
        
        try:
            # 1. Extract five raw DataFrames
            raw = extract_awards(year)

            # 2. Clean five DataFrames
            cleaned = clean_awards(raw, year)

            # 3. Save both raw and cleaned tables
            for award in award_names:

                raw[award].to_csv(
                    raw_folder / f"{award}.csv",
                    index=False,
                    encoding="utf-8"
                )

                cleaned[award].to_csv(
                    staging_folder / f"{award}.csv",
                    index=False,
                    encoding="utf-8"
                )

                # Collect cleaned DataFrames
                cleaned_history[award].append(cleaned[award])

            print(f"Completed {year}")

        except Exception as e:
            failed_years.append(year)
            print(f"Failed {year}: {e}")

        # Pause between requests
        if year < end_year:
            time.sleep(5)

    # 4. Concatenate all successfully cleaned seasons
    historical_awards = {}

    output_folder = PROJECT_ROOT / "data" / "staging" / "awards"
    output_folder.mkdir(parents=True, exist_ok=True)

    for award in award_names:

        if cleaned_history[award]:

            historical_awards[award] = pd.concat(
                cleaned_history[award],
                ignore_index=True
            )

            historical_awards[award].to_csv(
                output_folder / f"{award}_history.csv",
                index=False,
                encoding="utf-8"
            )

    print("Finished processing.")
    print("Failed years:", failed_years)

    return historical_awards, failed_years


In [167]:
historical_awards, failed_years = build_awards_history(
    start_year=2001,
    end_year=2002
)

Processing 2001...
Completed 2001
Processing 2002...
Completed 2002
Finished processing.
Failed years: []


In [168]:
historical_awards

{'mvp':     voting_rank       player_name  age team  first_place_votes  points_won  \
 0             1     Allen Iverson   25  PHI                 93        1121   
 1             2        Tim Duncan   24  SAS                 18         706   
 2             3  Shaquille O'Neal   28  LAL                  7         578   
 3             4      Chris Webber   27  SAC                  5         521   
 4             5     Kevin Garnett   24  MIN                  1         151   
 5             6     Tracy McGrady   21  ORL                  0          64   
 6             7       Karl Malone   37  UTA                  0          21   
 7             8        Jason Kidd   27  PHO                  0          18   
 8             9       Kobe Bryant   22  LAL                  0          11   
 9            10    David Robinson   35  SAS                  0           8   
 10           11         Ray Allen   25  MIL                  0           7   
 11           11      Vince Carter   24  TOR 

In [169]:
for award, df in historical_awards.items():
    print(award, df.shape, df["year"].unique())

mvp (35, 11) [2001 2002]
roy (10, 11) [2001 2002]
dpoy (18, 11) [2001 2002]
rookie (19, 11) [2001 2002]
defense (20, 10) [2001 2002]


In [172]:
historical_awards["dpoy"].head()

,voting_rank,player_name,age,team,first_place_votes,points_won,points_max,vote_share,season,year,award_winner
0,1,Dikembe Mutombo,34,TOT,48,48,123,0.390,2000-2001,2001,True
1,2,Kevin Garnett,24,MIN,26,26,123,0.211,2000-2001,2001,False
2,3,Tim Duncan,24,SAS,14,14,123,0.114,2000-2001,2001,False
3,4,Doug Christie,30,SAC,9,9,123,0.073,2000-2001,2001,False
4,5,David Robinson,35,SAS,6,6,123,0.049,2000-2001,2001,False


In [173]:
historical_awards, failed_years = build_awards_history(
    start_year=2001,
    end_year=2026
)

Processing 2001...
Completed 2001
Processing 2002...
Completed 2002
Processing 2003...
Completed 2003
Processing 2004...
Completed 2004
Processing 2005...
Completed 2005
Processing 2006...
Completed 2006
Processing 2007...
Completed 2007
Processing 2008...
Completed 2008
Processing 2009...
Completed 2009
Processing 2010...
Completed 2010
Processing 2011...
Completed 2011
Processing 2012...
Completed 2012
Processing 2013...
Completed 2013
Processing 2014...
Completed 2014
Processing 2015...
Completed 2015
Processing 2016...
Completed 2016
Processing 2017...
Completed 2017
Processing 2018...
Completed 2018
Processing 2019...
Completed 2019
Processing 2020...
Completed 2020
Processing 2021...
Completed 2021
Processing 2022...
Completed 2022
Processing 2023...
Completed 2023
Processing 2024...
Completed 2024
Processing 2025...
Completed 2025
Processing 2026...
Completed 2026
Finished processing.
Failed years: []


In [174]:
from pathlib import Path

award_names = ["mvp", "roy", "dpoy", "rookie", "defense"]
expected_years = set(range(2001, 2027))

print("FAILED YEARS:", failed_years)
print()

for award in award_names:

    df = historical_awards.get(award)

    if df is None:
        print(f"{award}: MISSING HISTORICAL DATAFRAME")
        continue

    actual_years = set(df["year"].dropna().astype(int))
    missing_years = sorted(expected_years - actual_years)

    raw_count = sum(
        (PROJECT_ROOT / "data" / "raw" / "awards" /
         str(year) / f"{award}.csv").is_file()
        for year in expected_years
    )

    staging_count = sum(
        (PROJECT_ROOT / "data" / "staging" / "awards" /
         str(year) / f"{award}.csv").is_file()
        for year in expected_years
    )

    history_exists = (
        PROJECT_ROOT / "data" / "staging" / "awards" /
        f"{award}_history.csv"
    ).is_file()

    print(f"AWARD: {award}")
    print("  Total rows:", len(df))
    print("  Seasons:", len(actual_years), "/ 26")
    print("  Missing seasons:", missing_years)
    print("  Raw CSVs:", raw_count, "/ 26")
    print("  Staging CSVs:", staging_count, "/ 26")
    print("  Historical CSV exists:", history_exists)
    print("  Duplicate rows:", df.duplicated().sum())
    print()

FAILED YEARS: []

AWARD: mvp
  Total rows: 352
  Seasons: 26 / 26
  Missing seasons: []
  Raw CSVs: 26 / 26
  Staging CSVs: 26 / 26
  Historical CSV exists: True
  Duplicate rows: 0

AWARD: roy
  Total rows: 198
  Seasons: 26 / 26
  Missing seasons: []
  Raw CSVs: 26 / 26
  Staging CSVs: 26 / 26
  Historical CSV exists: True
  Duplicate rows: 0

AWARD: dpoy
  Total rows: 426
  Seasons: 26 / 26
  Missing seasons: []
  Raw CSVs: 26 / 26
  Staging CSVs: 26 / 26
  Historical CSV exists: True
  Duplicate rows: 0

AWARD: rookie
  Total rows: 260
  Seasons: 26 / 26
  Missing seasons: []
  Raw CSVs: 26 / 26
  Staging CSVs: 26 / 26
  Historical CSV exists: True
  Duplicate rows: 0

AWARD: defense
  Total rows: 263
  Seasons: 26 / 26
  Missing seasons: []
  Raw CSVs: 26 / 26
  Staging CSVs: 26 / 26
  Historical CSV exists: True
  Duplicate rows: 0

